In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
import logging
import os
from datetime import datetime

#targetpath = "/Volumes/workspace/default/employees/deltaoutput/"
#sourcepath = "/Volumes/workspace/default/employees/inputfiles/"
#archivepath = "/Volumes/workspace/default/employees/archivefiles/"
#logspath = "/Volumes/workspace/default/employees/logfiles/"

#prod

params ={
    "source_path": "",
    "target_path": "",
    "archive_path":"",
    "logs_path":""
}

for k,v in params.items():
    dbutils.widgets.text(k,v)

sourcepath = dbutils.widgets.get("source_path")
targetpath = dbutils.widgets.get("target_path")
archivepath = dbutils.widgets.get("archive_path") 
logspath = dbutils.widgets.get("logs_path")

#logging 
logs_file = os.path.join(logspath,
                         f"{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}_employeesload.log")
logging.basicConfig(format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',\
                    filename=logs_file,level=logging.DEBUG,force=True)

logging.info("Job Started")

logging.debug(f"sourcepath: {sourcepath}")
logging.debug(f"targetpath: {targetpath}")
logging.debug(f"archivepath: {archivepath}")
logging.debug(f"logs_path: {logspath}")


files = dbutils.fs.ls(sourcepath)

files = [f for f in files if f.path.endswith(".csv")]

sorted_files = sorted(files,key=lambda x:x.modificationTime,reverse=False)

#spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

if sorted_files:
    logging.debug(f"Files are available in the path: {sourcepath}")

    for f in sorted_files:

        logging.debug(f"{f.name} is going to be loaded")
        if not DeltaTable.isDeltaTable(spark, targetpath):
            
            logging.debug(f"DeltaTable is not available in the path: {targetpath}")
            logging.info("Initial load")

            df = spark.read.format('csv').options(header='true', inferSchema='true').load(f.path)
            #df.show()
            logging.debug(f" Rows loaded: {df.count()}")
            df.coalesce(1).write.format("delta").mode("overwrite").save(targetpath)
            #spark.sql(f)
            try:
                dbutils.fs.mv(f.path,archivepath)
                logging.debug(f"{f.name} is archived in path: {archivepath}")
            except Exception as e:
                logging.debug(f"Exception occured while moving {f.name} to archive path : {e}")
                
        else:
            logging.debug(f"Delta Table is already available in the {targetpath}")
            logging.info(f"Upsert Starts Here")

            df = spark.read.format("delta").load(targetpath)
            logging.debug("Number of rows in Delta Table: {df.count}")
            cols = set(df.columns)

            logging.info("SCD Type 2 Starts Here")

            if "last_modified" not in cols:

                    logging.info("SCD Column last_modified is not found in Delta Table")
                    try:
                        spark.sql(f"""
                        ALTER TABLE delta.`{targetpath}`
                        ADD COLUMNS (last_modified TIMESTAMP)
                        """)
                        spark.sql(f"""
                        UPDATE delta.`{targetpath}`
                        SET last_modified = current_timestamp()
                        WHERE last_modified IS NULL
                        """)
                        logging.info("last_modified column is added to Delta Table and defaulted to current_timestamp()")
                    except Exception as e:
                        logging.debug("Error while adding SCD column last_modified: {e}")

            if "isactive" not in cols:
                logging.info("SCD Column last_modified is not found in Delta Table")
                try:
                    spark.sql(f"""
                    ALTER TABLE delta.`{targetpath}`
                    ADD COLUMNS (isactive INT)
                    """)
                    spark.sql(f"""
                    UPDATE delta.`{targetpath}`
                    SET isactive = 1
                    WHERE isactive IS NULL
                    """)
                    logging.info("isactive column is added to Delta Table and defaulted to 1")
                except Exception as e:
                    logging.debug("Error while adding SCD column isactive: {e}")


            
            target_df = spark.read.format("delta").load(targetpath)

            logging.debug(f"Total rows from Delta Table: {target_df.count}")
            target_df = target_df.filter("isactive=1")

            logging.debug(f"Active rows from Delta table: {target_df.count}")

            #target_df.printSchema()

            schema = spark.read.format("delta").load(targetpath).schema
            
            source_df = spark.read.format("csv").schema(schema).option("header",True).load(f.path)

            logging.debug(f"Rows from the source file {f.name} : {source_df.count}")

            source_added_columns = source_df.withColumn("last_modified", F.current_timestamp())\
                .withColumn("isactive",F.lit(1))
            
            '''source_added_columns = source_added_columns.withColumn("emp_id",F.col("emp_id").cast("int"))\
                .withColumn("emp_name",F.col("emp_name").cast("string"))\
                .withColumn("salary",F.col("salary").cast("int"))\
                .withColumn("address",F.col("address").cast("string"))\
                .withColumn("gender",F.col("gender").cast("string"))'''


            #source_added_columns.printSchema()

            joined= source_added_columns.alias("s").join(target_df.alias("t"), on="emp_id", how="left")

            #print("joined")
            #joined.show()

            new_records = joined.filter(F.col("t.Emp_id").isNull()).select("s.*")
            #print("new_records")
            #new_records.show()

            logging.debug(f"Count of new records after comparing: {new_records.count}")

            columns_except = [col for col in source_added_columns.columns if col not in ("last_modified","isactive")]

            change_condition = " OR ".join([f"s.{col} != t.{col}" for col in columns_except])

            #print("changed_condition")
            #print(change_condition)

            updated_records = joined.filter(F.col("t.Emp_id").isNotNull() & F.expr(change_condition)).select("s.*")

            logging.debug(f"Count of Changed Records: {updated_records.count}")
            #print("updated_records")
            #updated_records.show()

            updated_records.createOrReplaceTempView("updated_records")

            spark.sql("""
            UPDATE delta.`/Volumes/workspace/default/employees/deltaoutput/`
            SET isactive = 0, last_modified = current_timestamp()
            WHERE emp_id IN (SELECT emp_id FROM updated_records)
            AND isactive = 1
            """)

            logging.info("Set Changed Records of current rows to inactive")

            '''delta_table = DeltaTable.forPath(spark,targetpath)

            delta_table = delta_table.toDF()

            print("delta_table")

            delta_table.show() '''

            union = new_records.unionByName(updated_records)

            logging.debug(f"Total Rows to be loaded: {union.count}")

            #print("union")
            #union.show()

            #union.show()
            #union.printSchema()

            logging.info("Writing Data to Delta Table")
            union.coalesce(1).write.format("delta").option("mergeSchema",True).mode("append").save(targetpath)

            logging.info("Data is loaded to Delta Table")
            delta_table = DeltaTable.forPath(spark,targetpath)

            delta_table = delta_table.toDF()
            logging.debug(f"Total Rows after loading is {delta_table.count}")

            try:
                dbutils.fs.mv(f.path,archivepath)
                logging.debug(f"{f.name} is archived : {archivepath}")
            except Exception as e:
                logging.info(f"Error while archiving the file: {e}")
            logging.info("Job Ended")
            logging.shutdown()

else:
    logging.debug(f"No files found in {sourcepath}")
    logging.info("Job Ended")
    logging.shutdown()